# Reversible Data Hiding in Encrypted Images via Multi-Band Integer Wavelet Embedding and a Bounded Lozi–Chebyshev Chaotic Map

Reference implementation of the scheme in the paper (Harshita and Poonam Bansal, *Pattern Analysis and Applications*, revised version).

**Content owner (sender):** fall-off-boundary (FOB) pre-processing → 5/3 integer wavelet transform → embedding order from LL → masking of the message with the data-hiding key $K_d$ → multi-level expansion/shift embedding in HL, LH and HH → inverse IWT with range check → encryption with the 2D-LCM under the encryption key $K_e$ and a fresh public nonce $N$.

**Receiver:** decryption → IWT → same order from LL → blind extraction → inverse IWT → undo the FOB clamp → exact cover; with $K_d$ also the message.

**How to use**
1. Choose the input in **Section 1**:
   - `INPUT_MODE = 'single'` and `IMAGE_PATH` for one image;
   - `INPUT_MODE = 'folder'` and `FOLDER_PATH` for every image in a folder;
   - `INPUT_MODE = 'demo'` to try the code on built-in test images.
2. Choose the message and the payload (random bits at a given net rate or at the maximum net rate, a text, a file, or a binary secret image).
3. Run all cells (*Runtime → Run all*). Results, images and a CSV table are written to `OUTPUT_DIR`.

Colour images are converted to 8-bit grey (ITU-R BT.601 luma); an odd number of rows or columns is cropped by one so that the IWT is defined.

## 1. Settings

In [ ]:
# ============================== INPUT ==============================
INPUT_MODE  = 'demo'                     # 'single' | 'folder' | 'demo'
IMAGE_PATH  = '/content/Boat.png'        # used when INPUT_MODE == 'single'
FOLDER_PATH = '/content/images'          # used when INPUT_MODE == 'folder'
RECURSIVE   = False                      # also search sub-folders of FOLDER_PATH
EXTENSIONS  = ('.png', '.bmp', '.tif', '.tiff', '.jpg', '.jpeg', '.pgm', '.ppm')
MOUNT_GOOGLE_DRIVE   = False             # True: mount Drive, then use paths such as '/content/drive/MyDrive/...'
UPLOAD_FROM_COMPUTER = False             # True (single mode): opens a file picker and uses the uploaded image
RESIZE_TO   = None                       # None keeps the size; e.g. 512 -> centre crop to a square and resize to 512 x 512
OUTPUT_DIR  = '/content/rdhei_results'

# ============================== MESSAGE ==============================
MESSAGE_SOURCE    = 'random'             # 'random' | 'text' | 'file' | 'binary_image'
NET_RATE          = 0.10                 # 'random' only: net payload in bpp (message bits / pixels), or 'max' for the maximum net capacity
MESSAGE_TEXT      = 'Patient ID 0042 | Chest X-ray | 2026-10-30'   # 'text'
MESSAGE_FILE      = '/content/secret.bin'                         # 'file' (any file; its bytes are embedded)
SECRET_IMAGE_PATH = '/content/cameraman.png'                      # 'binary_image' (binarised at 128, as in the paper)
SECRET_IMAGE_SIZE = (256, 256)                                    # 256 x 256 = 65,536 bits as in the paper

# ============================== EMBEDDING ==============================
LEVEL = 'auto'                           # 'auto' or an integer 1..15. 'auto' follows the paper (Section 5.4): the two smallest
                                         # feasible levels x the exclusion settings, keep the highest PSNR. With NET_RATE = 'max' the
                                         # level of maximum net capacity is used (scans L = 1..15, about 1-3 min per 512 x 512 image)
RHOS  = (2.0, 0.5, 0.25, 0.1, 0.05)      # exclusion thresholds rho; 2.0 means no exclusion map

# ============================== KEYS ==============================
KE = (0.3, 0.6, 5.0, 20.7, 2.3)          # encryption key  (x0, y0, a, b, w)  -- working key of the paper
KD = (-0.41, 0.27, 6.1, 18.3, 3.7)       # data-hiding key (x0, y0, a, b, w)  -- independent of KE
NONCE_MODE = 'random'                    # 'random': fresh 128-bit nonce per image (required in practice)
                                         # 'derived': nonce derived from the file name (reproducible runs)

# ============================== ANALYSIS ==============================
SECURITY_METRICS = True                  # entropy, local entropy, chi-square, correlation, NPCR/UACI
NPCR_TRIALS      = 3                     # one-pixel changes of the cover per image (paper: 10)
SAVE_FIGURES     = True
SHOW_FIGURES     = True

## 2. Environment

In [ ]:
import os, sys, glob, json, math, time, zlib, ctypes, subprocess, secrets, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from PIL import Image
try:
    from skimage.metrics import structural_similarity
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-image'], check=True)
    from skimage.metrics import structural_similarity
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB and MOUNT_GOOGLE_DRIVE:
    from google.colab import drive; drive.mount('/content/drive')
os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Python', sys.version.split()[0], '| NumPy', np.__version__, '| Colab' if IN_COLAB else '| local')

## 3. 2D-LCM and cipher kernel

The 2D-LCM is iterated in IEEE-754 double precision without fast-math optimisations; this is part of the specification (Section 5.7.7 of the paper). The cell compiles the reference C kernel with `gcc` (available in Colab). If no compiler is available, an identical pure-Python version is used (same operations in the same order, about 20 times slower).

In [ ]:
C_SOURCE = r'''
/* Fast kernels for the 2D Lozi-Chebyshev map (2D-LCM) and the chained cipher.
 * Bounded ("fold") realisation used throughout the revised manuscript:
 *     x_{n+1} = W( 1 - a*|cos(w*acos(x_n))| + y_n ),   W(u) = u+1 - 2*floor((u+1)/2) - 1  (fold onto [-1,1))
 *     y_{n+1} = b * x_n
 *     p_{n+1} = W( x_{n+1} + y_{n+1} )
 * Compile: gcc -O2 -shared -fPIC -o liblcm.so lcm_core.c -lm      (no -ffast-math: IEEE-754 semantics required)
 */
#include <math.h>
#include <stdint.h>
#include <string.h>

static inline double wrap11(double u) { double v = u + 1.0; return v - 2.0 * floor(v / 2.0) - 1.0; }

static inline void step(double *x, double *y, double a, double b, double w) {
    double xc = *x, yc = *y;
    double xg = xc > 1.0 ? 1.0 : (xc < -1.0 ? -1.0 : xc);
    double xn = wrap11(1.0 - a * fabs(cos(w * acos(xg))) + yc);
    *y = b * xc; *x = xn;
}

/* Key schedule: absorb an optional nonce (nb bytes) into the initial state, then discard `transient`
 * iterates. Absorption of byte u: x <- W(x + (u+1)/257), followed by 8 map iterations. */
void lcm_init(double *x, double *y, double a, double b, double w,
              const uint8_t *nonce, int nb, int64_t transient) {
    for (int j = 0; j < nb; j++) {
        *x = wrap11(*x + ((double)nonce[j] + 1.0) / 257.0);
        for (int r = 0; r < 8; r++) step(x, y, a, b, w);
    }
    for (int64_t k = 0; k < transient; k++) step(x, y, a, b, w);
}

/* Generate n iterates (x, y, p) starting from state (x0,y0) (state is NOT advanced past the output). */
void lcm_gen(double x0, double y0, double a, double b, double w, int64_t n,
             double *xs, double *ys, double *ps) {
    double x = x0, y = y0;
    for (int64_t k = 0; k < n; k++) {
        step(&x, &y, a, b, w);
        xs[k] = x; ys[k] = y; ps[k] = wrap11(x + y);
    }
}

/* Byte keystream from p: floor(|p| * 2^32) mod 256 (bits 25..32 below the binary point). */
void bytes_from_p(const double *p, int64_t n, uint8_t *out) {
    for (int64_t k = 0; k < n; k++) out[k] = (uint8_t)(((uint64_t)floor(fabs(p[k]) * 4294967296.0)) & 0xFF);
}

/* One cipher round: c_t = ((v_t + c_{t-1}) mod 256) XOR k_t,   c_{-1} = s. */
void chain_fwd(const uint8_t *v, const uint8_t *k, uint8_t s, int64_t n, uint8_t *c) {
    uint8_t prev = s;
    for (int64_t t = 0; t < n; t++) { prev = (uint8_t)((uint8_t)(v[t] + prev) ^ k[t]); c[t] = prev; }
}
void chain_inv(const uint8_t *c, const uint8_t *k, uint8_t s, int64_t n, uint8_t *v) {
    uint8_t prev = s;
    for (int64_t t = 0; t < n; t++) { v[t] = (uint8_t)((c[t] ^ k[t]) - prev); prev = c[t]; }
}
'''

KDIR = '/tmp/rdhei_kernel'; os.makedirs(KDIR, exist_ok=True)
open(f'{KDIR}/lcm_core.c', 'w').write(C_SOURCE)
import shutil
FORCE_PURE_PYTHON = False                # True: skip the compiler (results are identical, only slower)
_cc = subprocess.run(['gcc', '-O2', '-shared', '-fPIC', '-o', f'{KDIR}/liblcm.so', f'{KDIR}/lcm_core.c', '-lm'], capture_output=True, text=True) \
      if shutil.which('gcc') and not FORCE_PURE_PYTHON else None
USE_C = _cc is not None and _cc.returncode == 0
if USE_C:
    _lib = ctypes.CDLL(f'{KDIR}/liblcm.so'); _D = ctypes.c_double; _I64 = ctypes.c_int64
    _PD = np.ctypeslib.ndpointer(np.float64, flags='C'); _PU8 = np.ctypeslib.ndpointer(np.uint8, flags='C')
    _lib.lcm_init.argtypes = [ctypes.POINTER(_D), ctypes.POINTER(_D), _D, _D, _D, ctypes.c_char_p, ctypes.c_int, _I64]
    _lib.lcm_gen.argtypes = [_D, _D, _D, _D, _D, _I64, _PD, _PD, _PD]
    _lib.bytes_from_p.argtypes = [_PD, _I64, _PU8]
    for f in ('chain_fwd', 'chain_inv'): getattr(_lib, f).argtypes = [_PU8, _PU8, ctypes.c_uint8, _I64, _PU8]
print('2D-LCM kernel:', 'compiled C (gcc -O2, IEEE-754)' if USE_C else 'pure Python (no compiler found)')

TRANSIENT = 2000

class Key:
    # Secret key K = (x0, y0, a, b, w) of the bounded 2D-LCM, Eq. (17); admissible domain Eq. (18).
    def __init__(self, x0, y0, a, b, w): self.x0, self.y0, self.a, self.b, self.w = map(float, (x0, y0, a, b, w))
    def tup(self): return (self.x0, self.y0, self.a, self.b, self.w)
    def check(self, name):
        ok = -1 <= self.x0 < 1 and -1 <= self.y0 < 1 and 1 <= self.a <= 20 and 5 <= self.b <= 25 and 2 <= self.w <= 10
        if not ok: warnings.warn(f'{name} = {self.tup()} lies outside the admissible key domain of Eq. (18)')
        return ok

# ---- pure-Python fallback: the same operations, in the same order, as the C kernel
def _wrap11(u): v = u + 1.0; return v - 2.0 * math.floor(v / 2.0) - 1.0
def _step(x, y, a, b, w):
    xg = 1.0 if x > 1.0 else (-1.0 if x < -1.0 else x)
    return _wrap11(1.0 - a * abs(math.cos(w * math.acos(xg))) + y), b * x

def lcm_sequence(K, n, nonce=b'', transient=TRANSIENT):
    # Key schedule, Eq. (19): absorb the nonce bytes, discard the transient, then generate x_n, y_n, p_n.
    if USE_C:
        x = _D(K.x0); y = _D(K.y0)
        _lib.lcm_init(ctypes.byref(x), ctypes.byref(y), K.a, K.b, K.w, nonce, len(nonce), transient)
        xs = np.empty(n); ys = np.empty(n); ps = np.empty(n)
        _lib.lcm_gen(x.value, y.value, K.a, K.b, K.w, n, xs, ys, ps); return xs, ys, ps
    x, y, a, b, w = K.x0, K.y0, K.a, K.b, K.w
    for u in nonce:
        x = _wrap11(x + (u + 1.0) / 257.0)
        for _ in range(8): x, y = _step(x, y, a, b, w)
    for _ in range(transient): x, y = _step(x, y, a, b, w)
    xs = np.empty(n); ys = np.empty(n); ps = np.empty(n)
    for k in range(n):
        x, y = _step(x, y, a, b, w); xs[k] = x; ys[k] = y; ps[k] = _wrap11(x + y)
    return xs, ys, ps

def p_bytes(p):
    # Eq. (20): key byte = floor(|p| * 2^32) mod 256.
    if USE_C:
        out = np.empty(len(p), np.uint8); _lib.bytes_from_p(np.ascontiguousarray(p), len(p), out); return out
    return (np.floor(np.abs(p) * 4294967296.0).astype(np.uint64) & 0xFF).astype(np.uint8)

def _chain_fwd(v, k, s):
    # One cipher round, Eq. (28): c_t = ((v_t + c_{t-1}) mod 256) XOR k_t, c_0 = s.
    if USE_C:
        out = np.empty(len(v), np.uint8); _lib.chain_fwd(np.ascontiguousarray(v, np.uint8), np.ascontiguousarray(k, np.uint8), int(s), len(v), out); return out
    out = np.empty(len(v), np.uint8); prev = int(s); vv = v.tolist(); kk = k.tolist()
    for t in range(len(vv)): prev = ((vv[t] + prev) & 255) ^ kk[t]; out[t] = prev
    return out

def _chain_inv(c, k, s):
    # Inverse round, Eq. (28): v_t = ((c_t XOR k_t) - c_{t-1}) mod 256.
    if USE_C:
        out = np.empty(len(c), np.uint8); _lib.chain_inv(np.ascontiguousarray(c, np.uint8), np.ascontiguousarray(k, np.uint8), int(s), len(c), out); return out
    out = np.empty(len(c), np.uint8); prev = int(s); cc = c.tolist(); kk = k.tolist()
    for t in range(len(cc)): out[t] = ((cc[t] ^ kk[t]) - prev) & 255; prev = cc[t]
    return out

## 4. Data-hiding layer (reference code, unchanged)

5/3 integer lifting (Eqs. 1–7), adaptive arithmetic coder, FOB pre-processing (Eqs. 23–24), receiver-reproducible embedding order (Eq. 25), multi-level expansion/shift embedding and blind extraction (Eqs. 26–27, Proposition 1), the two-stage embedded stream with the 76-bit header (Table 7) and the margin search.

In [ ]:
# ----------------------------------------------------------------------------- 5/3 integer lifting (JPEG2000 reversible)
def _lift53(X):                      # along axis 0
    Xe = X[0::2]; Xo = X[1::2]
    Xe_n = np.concatenate([Xe[1:], Xe[-1:]], 0)
    D = Xo - np.floor_divide(Xe + Xe_n, 2)
    D_p = np.concatenate([D[:1], D[:-1]], 0)
    S = Xe + np.floor_divide(D_p + D + 2, 4)
    return S, D

def _ilift53(S, D):
    D_p = np.concatenate([D[:1], D[:-1]], 0)
    Xe = S - np.floor_divide(D_p + D + 2, 4)
    Xe_n = np.concatenate([Xe[1:], Xe[-1:]], 0)
    Xo = D + np.floor_divide(Xe + Xe_n, 2)
    X = np.empty((2 * S.shape[0],) + S.shape[1:], dtype=np.int64); X[0::2] = Xe; X[1::2] = Xo
    return X

def iwt53(X):
    """Returns ll, hl, lh, hh (columns first, then rows; identical ordering to the released MATLAB code)."""
    S, D = _lift53(np.asarray(X, np.int64))
    ll, hl = (t.T for t in _lift53(S.T)); lh, hh = (t.T for t in _lift53(D.T))
    return ll, hl, lh, hh

def iiwt53(ll, hl, lh, hh):
    S = _ilift53(ll.T, hl.T).T; D = _ilift53(lh.T, hh.T).T
    return _ilift53(S, D)

# ----------------------------------------------------------------------------- adaptive arithmetic coder (32-bit, WNC)
class _AC:
    TOP = (1 << 32) - 1; HALF = 1 << 31; Q1 = 1 << 30; Q3 = 3 << 30

def ac_encode(symbols, nsym):
    lo, hi, pend, out = 0, _AC.TOP, 0, []
    freq = [1] * nsym; tot = nsym
    def emit(b):
        nonlocal pend
        out.append(b); out.extend([1 - b] * pend); pend = 0
    for s in symbols:
        rng = hi - lo + 1; cum = sum(freq[:s])
        hi = lo + rng * (cum + freq[s]) // tot - 1; lo = lo + rng * cum // tot
        while True:
            if hi < _AC.HALF: emit(0)
            elif lo >= _AC.HALF: emit(1); lo -= _AC.HALF; hi -= _AC.HALF
            elif lo >= _AC.Q1 and hi < _AC.Q3: pend += 1; lo -= _AC.Q1; hi -= _AC.Q1
            else: break
            lo = 2 * lo; hi = 2 * hi + 1
        freq[s] += 1; tot += 1
        if tot > 65535: freq = [(f + 1) // 2 for f in freq]; tot = sum(freq)
    pend += 1; emit(0 if lo < _AC.Q1 else 1)
    return np.array(out, np.uint8)

def ac_decode(bits, n, nsym):
    bits = list(map(int, bits)); pos = 0
    def nb():
        nonlocal pos
        b = bits[pos] if pos < len(bits) else 0; pos += 1; return b
    lo, hi, val = 0, _AC.TOP, 0
    for _ in range(32): val = 2 * val + nb()
    freq = [1] * nsym; tot = nsym; out = []
    for _ in range(n):
        rng = hi - lo + 1; tgt = ((val - lo + 1) * tot - 1) // rng
        cum = 0; s = 0
        while cum + freq[s] <= tgt: cum += freq[s]; s += 1
        hi = lo + rng * (cum + freq[s]) // tot - 1; lo = lo + rng * cum // tot
        while True:
            if hi < _AC.HALF: pass
            elif lo >= _AC.HALF: lo -= _AC.HALF; hi -= _AC.HALF; val -= _AC.HALF
            elif lo >= _AC.Q1 and hi < _AC.Q3: lo -= _AC.Q1; hi -= _AC.Q1; val -= _AC.Q1
            else: break
            lo = 2 * lo; hi = 2 * hi + 1; val = 2 * val + nb()
        out.append(s); freq[s] += 1; tot += 1
        if tot > 65535: freq = [(f + 1) // 2 for f in freq]; tot = sum(freq)
    return np.array(out, np.int64)

# ----------------------------------------------------------------------------- helpers
def int2bits(v, n): return np.array([(int(v) >> (n - 1 - k)) & 1 for k in range(n)], np.uint8)
def bits2int(b): return int(''.join(str(int(x)) for x in b), 2) if len(b) else 0

# ----------------------------------------------------------------------------- FOB handling
# Coefficient (m,n) of any detail band can only alter pixels in rows/cols [2m-1, 2m+3] x [2n-1, 2n+3] (measured
# support of the 5/3 synthesis); one pixel of safety margin is added on each side.
INF_LO, INF_HI = -2, 4
BLK = 8                       # exclusion-map granularity: 8x8 coefficient positions = 16x16 pixels
SAT_T = 16                    # ordering only: positions whose 3x3 LL neighbourhood touches [0,16] or [239,255] go last

def influence(active, shape):
    """Pixel mask R that can be altered by modifying coefficients at the active positions."""
    M, N = shape; R = np.zeros((M, N), bool); ms, ns = np.nonzero(active)
    for dy in range(INF_LO, INF_HI + 1):
        for dx in range(INF_LO, INF_HI + 1):
            r = 2 * ms + dy; c = 2 * ns + dx; ok = (r >= 0) & (r < M) & (c >= 0) & (c < N); R[r[ok], c[ok]] = True
    return R

def blocks_to_positions(E, pshape):
    return np.kron(E.astype(np.uint8), np.ones((BLK, BLK), np.uint8)).astype(bool)[:pshape[0], :pshape[1]]

def choose_exclusion(I, D, rho):
    """Sender's (free) choice of excluded 16x16 pixel blocks: blocks whose fraction of near-boundary pixels
    (<= D or >= 255-D) exceeds rho. rho > 1 disables exclusion."""
    M, N = I.shape; nb = ((M // 2 + BLK - 1) // BLK, (N // 2 + BLK - 1) // BLK)
    if rho > 1: return np.zeros(nb, bool)
    near = ((I <= D) | (I >= 255 - D)).astype(np.float64)
    s = 2 * BLK; E = np.zeros(nb, bool)
    for i in range(nb[0]):
        for j in range(nb[1]):
            E[i, j] = near[i * s:(i + 1) * s, j * s:(j + 1) * s].mean() > rho
    return E

def fob_map(I, Ip, R, D):
    amb = R & ((Ip == D) | (Ip == 255 - D))
    corr = np.abs(np.asarray(I, np.int64) - Ip)[amb]
    return (ac_encode(corr.tolist(), D + 1) if corr.size else np.zeros(0, np.uint8)), int(amb.sum())

def fob_restore(Ip, D, BF, R):
    amb = R & ((Ip == D) | (Ip == 255 - D)); n = int(amb.sum()); I = Ip.copy()
    if n:
        corr = ac_decode(BF, n, D + 1); sign = np.where(Ip[amb] == D, -1, 1); I[amb] = Ip[amb] + sign * corr
    return I

# ----------------------------------------------------------------------------- data hiding (plaintext IWT domain)
HDR_BITS = 76                 # L(4) | D(8) | len_E(16) | len_F(24) | len_M(24)
BAND_ORDER = {'all3': ('hh', 'hl', 'lh'), 'hh': ('hh',), 'hl+lh': ('hl', 'lh')}

def embedding_order(ll, bands='all3'):
    """All positions sorted by (saturation flag, local LL complexity): smooth, non-saturated first. A function of LL
    only; LL is never modified, so the receiver recomputes exactly the same order."""
    P = np.pad(ll, 1, mode='edge').astype(np.int64); h, w = ll.shape
    nb8 = [P[1 + dy:h + 1 + dy, 1 + dx:w + 1 + dx] for dy in (-1, 0, 1) for dx in (-1, 0, 1)]
    cx = sum(np.abs(P[1:-1, 1:-1] - q) for q in nb8)
    sat = (np.min(nb8, axis=0) <= SAT_T) | (np.max(nb8, axis=0) >= 255 - SAT_T)
    pos = np.lexsort((np.arange(cx.size), cx.ravel(), sat.ravel()))
    names = BAND_ORDER[bands]; nb = len(names)
    band_id = np.tile(np.arange(nb), len(pos)); flat = np.repeat(pos, nb)
    return names, band_id, flat

def _gather(bands_dict, names, band_id, flat):
    stack = np.stack([bands_dict[n].ravel() for n in names]); return stack[band_id, flat]

def _scatter(bands_dict, names, band_id, flat, vals):
    for k, n in enumerate(names):
        m = band_id == k; arr = bands_dict[n].ravel().copy(); arr[flat[m]] = vals[m]; bands_dict[n] = arr.reshape(bands_dict[n].shape)

def _expand(c, bits, L):
    """Embed bits sequentially into coefficient vector c at level L. Returns (c', n_used_coeffs)."""
    c = c.copy(); nbits = len(bits)
    if nbits == 0: return c, 0
    emb = (c >= -L) & (c <= L - 1); cs = np.cumsum(emb)
    if len(cs) == 0 or cs[-1] < nbits: raise ValueError('capacity')
    stop = int(np.searchsorted(cs, nbits)) + 1        # number of coefficients touched
    seg = c[:stop]; e = emb[:stop]; out = seg.copy()
    out[e] = 2 * seg[e] + bits
    out[(~e) & (seg >= L)] = seg[(~e) & (seg >= L)] + L
    out[(~e) & (seg < -L)] = seg[(~e) & (seg < -L)] - L
    c[:stop] = out
    return c, stop

def _extract(cw, nbits, L):
    """Inverse of _expand on the prefix; returns (bits, restored prefix, n_used_coeffs)."""
    if nbits == 0: return np.zeros(0, np.uint8), cw[:0], 0
    emb = (cw >= -2 * L) & (cw <= 2 * L - 1); cs = np.cumsum(emb)
    if len(cs) == 0 or cs[-1] < nbits: raise ValueError('stream shorter than header')
    stop = int(np.searchsorted(cs, nbits)) + 1
    seg = cw[:stop]; e = emb[:stop]
    bits = (seg[e] & 1).astype(np.uint8)
    rest = seg.copy(); rest[e] = seg[e] >> 1               # floor(c'/2)
    rest[(~e) & (seg >= 2 * L)] -= L; rest[(~e) & (seg < -2 * L)] += L
    return bits, rest, stop

def _embed_once(I, payload_bits, L, D, E, bands):
    """One attempt with margin D and exclusion map E. Returns (Im or None, info, E_violation_blocks)."""
    I = np.asarray(I, np.int64); M, N = I.shape; pshape = (M // 2, N // 2)
    Epos = blocks_to_positions(E, pshape)
    R = influence(~Epos, I.shape)
    Ip = np.where(R, np.clip(I, D, 255 - D), I)
    BF, namb = fob_map(I, Ip, R, D)
    BE = ac_encode(E.ravel().astype(int).tolist(), 2) if E.any() else np.zeros(0, np.uint8)
    ll, hl, lh, hh = iwt53(Ip); B = {'hl': hl, 'lh': lh, 'hh': hh}
    names, bid, flat = embedding_order(ll, bands)
    c = _gather(B, names, bid, flat)
    hdr = np.concatenate([int2bits(L, 4), int2bits(D, 8), int2bits(len(BE), 16), int2bits(len(BF), 24),
                          int2bits(len(payload_bits), 24)])
    # stage 1: header + exclusion map at level 1 along the full order (receiver does not know E yet)
    c1, s1 = _expand(c, np.concatenate([hdr, BE]).astype(np.int64), 1)
    viol = Epos.ravel()[flat[:s1]]
    if viol.any():                                   # stage-1 coefficients must lie outside excluded blocks
        bad = np.unique(flat[:s1][viol]); bm = np.zeros(E.shape, bool)
        bm[(bad // pshape[1]) // BLK, (bad % pshape[1]) // BLK] = True
        return None, None, bm
    # stage 2: FOB map + payload at level L along the order restricted to non-excluded positions
    keep = np.nonzero(~Epos.ravel()[flat[s1:]])[0] + s1
    tail, s2 = _expand(c1[keep], np.concatenate([BF, payload_bits]).astype(np.int64), L)
    cw = c1.copy(); cw[keep] = tail
    _scatter(B, names, bid, flat, cw)
    Im = iiwt53(ll, B['hl'], B['lh'], B['hh'])
    info = dict(L=L, D=D, len_E=len(BE), n_excl_blocks=int(E.sum()), len_F=len(BF), n_amb=namb,
                len_M=len(payload_bits), coeffs_used=int(s1 + s2), total_coeffs=len(c), R_frac=float(R.mean()))
    if Im.min() < 0 or Im.max() > 255: return None, info, None
    assert np.array_equal(Im[~R], Ip[~R])                 # nothing outside R is ever modified
    return Im.astype(np.uint8), info, None

def embed(I, payload_bits, L, bands='all3', D=None, rho=2.0, max_D=127):
    """Returns marked image Im and info. payload_bits are already Kd-encrypted message bits.
    D starts at 3L and is enlarged until the sender-side range check passes (D is carried in the header).
    rho controls the optional exclusion of saturated blocks (rho > 1: none)."""
    payload_bits = np.asarray(payload_bits, np.uint8)
    D = 3 * L if D is None else D
    I = np.asarray(I, np.int64)
    while D <= max_D:
        E = choose_exclusion(I, D, rho)
        for _ in range(64):
            Im, info, bm = _embed_once(I, payload_bits, L, D, E, bands)
            if bm is None: break
            E = E & ~bm                                      # un-exclude blocks hit by stage 1 (monotone)
        if Im is not None: return Im, info
        D += 1
    raise RuntimeError('no valid margin')

def extract(Im, bands='all3'):
    """Blind extraction + exact recovery. Returns (payload_bits, recovered cover, header dict)."""
    Im = np.asarray(Im, np.int64); M, N = Im.shape; pshape = (M // 2, N // 2)
    ll, hl, lh, hh = iwt53(Im); B = {'hl': hl, 'lh': lh, 'hh': hh}
    names, bid, flat = embedding_order(ll, bands)
    cw = _gather(B, names, bid, flat)
    hb, _, _ = _extract(cw, HDR_BITS, 1)
    L = bits2int(hb[:4]); D = bits2int(hb[4:12]); lE = bits2int(hb[12:28]); lF = bits2int(hb[28:52]); lM = bits2int(hb[52:76])
    sb, r1, s1 = _extract(cw, HDR_BITS + lE, 1)
    nbk = ((pshape[0] + BLK - 1) // BLK, (pshape[1] + BLK - 1) // BLK)
    E = ac_decode(sb[HDR_BITS:], nbk[0] * nbk[1], 2).reshape(nbk).astype(bool) if lE else np.zeros(nbk, bool)
    Epos = blocks_to_positions(E, pshape)
    keep = np.nonzero(~Epos.ravel()[flat[s1:]])[0] + s1
    rb, r2, s2 = _extract(cw[keep], lF + lM, L)
    c = cw.copy(); c[:s1] = r1; c[keep[:s2]] = r2
    _scatter(B, names, bid, flat, c)
    Ip = iiwt53(ll, B['hl'], B['lh'], B['hh'])
    R = influence(~Epos, Ip.shape)
    I = fob_restore(Ip, D, rb[:lF], R)
    return rb[lF:], I.astype(np.uint8), dict(L=L, D=D, len_E=lE, len_F=lF, len_M=lM)

RHOS = (2.0, 0.5, 0.25, 0.1, 0.05)

def max_net_capacity(I, L, bands='all3', rhos=RHOS):
    """Exact maximal NET payload (bits) at level L, every overhead deducted, maximised over the exclusion
    threshold rho, and verified by an actual full-load embedding."""
    I = np.asarray(I, np.int64)
    best = dict(net=0, gross=0, D=3 * L, len_E=0, len_F=0, n_amb=0, rho=None, n_excl_blocks=0,
                gross_ceiling=(3 if bands == 'all3' else 1) * I.size // 4)
    for rho in rhos:
        D = 3 * L
        while D <= 127:
            E = choose_exclusion(I, D, rho); pshape = (I.shape[0] // 2, I.shape[1] // 2)
            try:
                for _ in range(64):
                    Im, info, bm = _embed_once(I, np.zeros(0, np.uint8), L, D, E, bands)
                    if bm is None: break
                    E = E & ~bm
            except ValueError:
                break
            if info is None: break
            # count capacity of stage 2 exactly
            Epos = blocks_to_positions(E, pshape)
            R = influence(~Epos, I.shape); Ip = np.where(R, np.clip(I, D, 255 - D), I)
            ll, hl, lh, hh = iwt53(Ip); Bd = {'hl': hl, 'lh': lh, 'hh': hh}
            names, bid, flat = embedding_order(ll, bands); c = _gather(Bd, names, bid, flat)
            n1 = HDR_BITS + info['len_E']; e1 = (c >= -1) & (c <= 0); cs = np.cumsum(e1)
            if cs[-1] < n1: break
            s1 = int(np.searchsorted(cs, n1)) + 1
            keep = np.nonzero(~Epos.ravel()[flat[s1:]])[0] + s1
            cap2 = int(((c[keep] >= -L) & (c[keep] <= L - 1)).sum())
            net = cap2 - info['len_F']
            if net <= 0: break
            try:
                Im2, inf2 = embed(I, np.zeros(net, np.uint8), L, bands, D=D, rho=rho, max_D=D)
                if inf2['len_F'] + net > cap2: break
                if net > best['net']:
                    best = dict(net=net, gross=n1 + cap2, D=D, len_E=inf2['len_E'], len_F=inf2['len_F'], n_amb=inf2['n_amb'],
                                rho=rho, n_excl_blocks=inf2['n_excl_blocks'], gross_ceiling=best['gross_ceiling'])
                break
            except (RuntimeError, ValueError):
                D += 1
    return best

## 5. Cipher and protocol

Key bytes, initial values and rank permutations from $K_e$ and the nonce (Eqs. 19–21); three chained rounds with two permutations (Eqs. 28–29); message mask from $K_d$ (Eq. 22).

In [ ]:
def _cipher_material(K, MN, nonce):
    xs, ys, ps = lcm_sequence(K, 3 * MN + 3, nonce)
    ks = [p_bytes(ps[r * MN:(r + 1) * MN]) for r in range(3)]                       # k_r(t), Eq. (20)
    perms = [np.argsort(xs[r * MN:(r + 1) * MN], kind='stable') for r in range(2)]   # pi_1, pi_2, Eq. (21): bijective
    seeds = p_bytes(ps[3 * MN:3 * MN + 3])                                          # s_1, s_2, s_3
    return ks, perms, seeds

def encrypt(Im, K, nonce):
    # Eq. (29): I_s = R3(pi_2(R2(pi_1(R1(v)))))
    shp = Im.shape; MN = Im.size; ks, perms, seeds = _cipher_material(K, MN, nonce)
    v = np.ascontiguousarray(Im.ravel().astype(np.uint8))
    for r in range(3):
        v = _chain_fwd(v, ks[r], seeds[r])
        if r < 2: v = np.ascontiguousarray(v[perms[r]])
    return v.reshape(shp)

def decrypt(E, K, nonce):
    shp = E.shape; MN = E.size; ks, perms, seeds = _cipher_material(K, MN, nonce)
    c = np.ascontiguousarray(E.ravel().astype(np.uint8))
    for r in (2, 1, 0):
        c = _chain_inv(c, ks[r], seeds[r])
        if r > 0:
            ip = np.empty(MN, np.int64); ip[perms[r - 1]] = np.arange(MN); c = np.ascontiguousarray(c[ip])
    return c.reshape(shp)

def payload_mask(Kd, n, nonce):
    # Eq. (22): one mask bit per iterate, the LSB of the key byte generated from K_d and the nonce.
    _, _, ps = lcm_sequence(Kd, n, nonce)
    return (p_bytes(ps) & 1).astype(np.uint8)

def sender(I, message_bits, L, Ke, Kd, nonce, bands='all3', rho=2.0):
    m = np.asarray(message_bits, np.uint8)
    em = m ^ payload_mask(Kd, len(m), nonce)
    Im, info = embed(I, em, L, bands, rho=rho)
    return encrypt(Im, Ke, nonce), Im, info

def receiver(E, Ke, Kd, nonce, bands='all3'):
    Im = decrypt(E, Ke, nonce)
    em, I, hdr = extract(Im, bands)
    m = em ^ payload_mask(Kd, len(em), nonce) if Kd is not None else None
    return m, I, Im, em, hdr

## 6. Self-test (runs in a few seconds)

In [ ]:
def self_test():
    rng = np.random.default_rng(0)
    for c in range(-60, 61):                                   # Proposition 1 (expanded and shifted cases), L = 1..6
        for L in range(1, 7):
            for b in (0, 1):
                arr = np.array([c, 0]); cw, s = _expand(arr, np.array([b]), L)      # c is expanded or shifted; 0 then carries b if c did not
                bb, rest, s2 = _extract(cw, 1, L); assert bb[0] == b and s2 == s and np.array_equal(rest, arr[:s])
    X = rng.integers(0, 256, (64, 48))
    assert np.array_equal(iiwt53(*iwt53(X)), X)                  # Eq. (7): exact integer IWT
    syms = rng.integers(0, 7, 500).tolist(); assert ac_decode(ac_encode(syms, 7), 500, 7).tolist() == syms
    K = Key(0.3, 0.6, 5.0, 20.7, 2.3); N = bytes(range(16)); P = rng.integers(0, 256, (32, 40), dtype=np.uint8)
    assert np.array_equal(decrypt(encrypt(P, K, N), K, N), P)
    # known-answer test against the reference implementation of the paper (same key, nonce 0..15)
    xs, ys, ps = lcm_sequence(K, 4, N)
    assert p_bytes(ps).tolist() == KAT_BYTES, (p_bytes(ps).tolist(), KAT_BYTES)
    print('self-test passed: Proposition 1, exact IWT, arithmetic coder, cipher inverse, 2D-LCM known-answer test')
KAT_BYTES = [246, 0, 140, 51]
self_test()

## 7. Input images and message

In [ ]:
def load_gray(path):
    im = Image.open(path)
    if im.mode in ('I;16', 'I;16B', 'I', 'F'):                     # 16-bit / float images: rescale to 8 bit
        A = np.asarray(im).astype(np.float64); A = (A - A.min()) / max(A.max() - A.min(), 1e-12) * 255; im = Image.fromarray(np.round(A).astype(np.uint8))
    im = im.convert('L')                                           # ITU-R BT.601 luma for colour images
    if RESIZE_TO:
        s = min(im.size); l, t = (im.width - s) // 2, (im.height - s) // 2
        im = im.crop((l, t, l + s, t + s)).resize((RESIZE_TO, RESIZE_TO), Image.LANCZOS)
    A = np.asarray(im, np.uint8)
    return A[:A.shape[0] // 2 * 2, :A.shape[1] // 2 * 2]          # even size for the IWT

def demo_images():
    from skimage import data
    d = os.path.join(OUTPUT_DIR, '_demo_images'); os.makedirs(d, exist_ok=True)
    for name, arr in (('camera', data.camera()), ('astronaut', data.astronaut()), ('moon', data.moon())):
        Image.fromarray(arr).save(os.path.join(d, name + '.png'))
    return d

def list_images():
    if INPUT_MODE == 'single':
        path = IMAGE_PATH
        if UPLOAD_FROM_COMPUTER and IN_COLAB:
            from google.colab import files
            up = files.upload(); path = os.path.join('/content', next(iter(up)))
        if not os.path.isfile(path): raise FileNotFoundError(f'IMAGE_PATH not found: {path}')
        return [path]
    folder = demo_images() if INPUT_MODE == 'demo' else FOLDER_PATH
    if not os.path.isdir(folder): raise FileNotFoundError(f'FOLDER_PATH not found: {folder}')
    pat = os.path.join(folder, '**', '*') if RECURSIVE else os.path.join(folder, '*')
    paths = sorted(p for p in glob.glob(pat, recursive=RECURSIVE) if p.lower().endswith(EXTENSIONS))
    if not paths: raise FileNotFoundError(f'no images with extensions {EXTENSIONS} in {folder}')
    return paths

def bytes_to_bits(b): return np.unpackbits(np.frombuffer(b, np.uint8)) if len(b) else np.zeros(0, np.uint8)
def bits_to_bytes(bits): return np.packbits(np.asarray(bits, np.uint8)).tobytes()

def message_for(name, n_pixels):
    # Returns (message bits, description). For 'random' with NET_RATE == 'max' the length is set later.
    if MESSAGE_SOURCE == 'random':
        if NET_RATE == 'max': return None, 'random bits at maximum net capacity'
        n = int(round(float(NET_RATE) * n_pixels))
        return np.random.default_rng(zlib.crc32(name.encode())).integers(0, 2, n).astype(np.uint8), f'{n} random bits ({float(NET_RATE):.3f} bpp)'
    if MESSAGE_SOURCE == 'text':
        return bytes_to_bits(MESSAGE_TEXT.encode('utf-8')), f'text, {len(MESSAGE_TEXT.encode())} bytes'
    if MESSAGE_SOURCE == 'file':
        b = open(MESSAGE_FILE, 'rb').read(); return bytes_to_bits(b), f'file {os.path.basename(MESSAGE_FILE)}, {len(b)} bytes'
    if MESSAGE_SOURCE == 'binary_image':
        S = np.asarray(Image.open(SECRET_IMAGE_PATH).convert('L').resize(SECRET_IMAGE_SIZE, Image.LANCZOS))
        return (S >= 128).astype(np.uint8).ravel(), f'binary image {SECRET_IMAGE_SIZE[0]}x{SECRET_IMAGE_SIZE[1]}'
    raise ValueError(MESSAGE_SOURCE)

def nonce_for(name):
    if NONCE_MODE == 'random': return secrets.token_bytes(16)
    return np.random.default_rng(zlib.crc32(name.encode())).integers(0, 256, 16, dtype=np.uint8).tobytes()

Ke, Kd = Key(*KE), Key(*KD); Ke.check('KE'); Kd.check('KD')
IMAGES = list_images()
print(f'{len(IMAGES)} image(s):', *[os.path.basename(p) for p in IMAGES[:10]], '...' if len(IMAGES) > 10 else '')

## 8. Content owner: choice of level and exclusion, embedding, encryption

The receiver needs no knowledge of these choices: $L$, $D$ and the stream lengths are carried in the 76-bit header and the exclusion map in $B_E$.

In [ ]:
def capacity_at_max(I):
    # Maximum net payload over L = 1..15 and the exclusion settings (Section 5.3), every overhead deducted.
    best = None
    for L in range(1, 16):
        r = max_net_capacity(I, L, rhos=RHOS)
        if r['net'] > 0 and (best is None or r['net'] > best[1]['net']): best = (L, r)
    if best is None: raise RuntimeError('the image has no positive net capacity')
    return best

def plan_and_send(I, m, nonce):
    # Returns (I_s, I_m, info, L, rho, m). Selection as in Section 5.4: the two smallest feasible levels x RHOS -> highest PSNR.
    if m is None:                                                   # 'max'
        L, r = capacity_at_max(I)
        m = np.random.default_rng(1).integers(0, 2, r['net']).astype(np.uint8); levels, rhos = [L], [r['rho']]
    else:
        levels = range(1, 16) if LEVEL in ('auto', 'max') else [int(LEVEL)]; rhos = RHOS
    em = m ^ payload_mask(Kd, len(m), nonce)                         # Eq. (22)
    best = None; feasible = 0
    for L in levels:
        ok_L = False
        for rho in rhos:
            try: Im, info = embed(I, em, L, rho=rho)
            except (ValueError, RuntimeError): continue
            ok_L = True; p = psnr(I, Im)
            if best is None or p > best[0]: best = (p, Im, info, L, rho)
        feasible += ok_L
        if feasible == 2 and LEVEL == 'auto': break
    if best is None: raise RuntimeError(f'payload of {len(m)} bits does not fit (try a smaller NET_RATE or message)')
    _, Im, info, L, rho = best
    Is = encrypt(Im, Ke, nonce)
    assert np.array_equal(Is, sender(I, m, L, Ke, Kd, nonce, rho=rho)[0])   # identical to the paper's sender()
    return Is, Im, info, L, rho, m

## 9. Metrics (definitions as in Section 5 of the paper)

In [ ]:
def psnr(A, B):
    e = np.mean((A.astype(np.float64) - B.astype(np.float64)) ** 2)
    return float('inf') if e == 0 else 10 * np.log10(255.0 ** 2 / e)                              # Eq. (31)
def mse(A, B): return float(np.mean((A.astype(np.float64) - B.astype(np.float64)) ** 2))
def ssim(A, B): return float(structural_similarity(A, B, data_range=255, gaussian_weights=True, sigma=1.5, use_sample_covariance=False))  # Eq. (32)
def nc(w, w2):                                                                                   # Eq. (33)
    w = w.astype(np.float64); w2 = w2.astype(np.float64); a = w - w.mean(); b = w2 - w2.mean()
    d = np.sqrt((a * a).sum() * (b * b).sum()); return float((a * b).sum() / d) if d else float(np.array_equal(w, w2))
def entropy(I):                                                                                  # Eq. (34)
    h = np.bincount(I.ravel(), minlength=256) / I.size; h = h[h > 0]; return float(-(h * np.log2(h)).sum())
def local_entropy(I, k=30, T=1936, seed=0):                                                      # Wu et al.: 30 blocks of 44 x 44
    rng = np.random.default_rng(seed); M, N = I.shape; s = int(np.sqrt(T))
    if M < s or N < s: return float('nan')
    return float(np.mean([entropy(I[i:i + s, j:j + s]) for i, j in ((rng.integers(0, M - s + 1), rng.integers(0, N - s + 1)) for _ in range(k))]))
def chi2(I): h = np.bincount(I.ravel(), minlength=256); e = I.size / 256; return float(((h - e) ** 2 / e).sum())
def corr_all(I):                                                                                 # Eq. (35), all adjacent pairs
    I = I.astype(np.float64); P = {'H': (I[:, :-1], I[:, 1:]), 'V': (I[:-1, :], I[1:, :]), 'D': (I[:-1, :-1], I[1:, 1:])}
    return {k: float(np.corrcoef(a.ravel(), b.ravel())[0, 1]) for k, (a, b) in P.items()}
def npcr_uaci(A, B):                                                                             # Eqs. (36)-(37)
    A = A.astype(np.float64); B = B.astype(np.float64); return 100 * np.mean(A != B), 100 * np.mean(np.abs(A - B)) / 255
def npcr_critical(MN, alpha=0.05):                                                               # Wu, Noonan and Agaian (2011)
    from scipy.stats import norm
    F = 255; z = norm.ppf(1 - alpha); zz = norm.ppf(1 - alpha / 2)
    n = (F - z * np.sqrt(F / MN)) / (F + 1) * 100
    mu = (F + 2) / (3 * F + 3) * 100; sd = np.sqrt((F + 2) * (F * F + 2 * F + 3) / (18 * (F + 1) ** 2 * MN * F)) * 100
    return n, (mu - zz * sd, mu + zz * sd)

## 10. Run: content owner → channel → receiver, with all checks

In [ ]:
def process(path):
    name = os.path.splitext(os.path.basename(path))[0]; out = os.path.join(OUTPUT_DIR, name); os.makedirs(out, exist_ok=True)
    I = load_gray(path); M, N = I.shape; MN = I.size
    m, mdesc = message_for(name, MN); nonce = nonce_for(name)
    t0 = time.time(); Is, Im, info, L, rho, m = plan_and_send(I, m, nonce); t_s = time.time() - t0
    # ---------------- transmitted: I_s and the public nonce N only
    Image.fromarray(I).save(f'{out}/cover_gray.png'); Image.fromarray(Im).save(f'{out}/marked.png')
    Image.fromarray(Is).save(f'{out}/encrypted.png'); open(f'{out}/nonce.hex', 'w').write(nonce.hex())
    # ---------------- receiver with K_e and K_d
    t0 = time.time(); m_rec, I_rec, Im_rec, em, hdr = receiver(Is, Ke, Kd, nonce); t_r = time.time() - t0
    Image.fromarray(I_rec).save(f'{out}/recovered.png')
    exact_m = len(m_rec) == len(m) and np.array_equal(m_rec, m); exact_I = np.array_equal(I_rec, I)
    # ---------------- receiver with K_e only: cover exact, message masked
    _, I_e, _, em_e, _ = receiver(Is, Ke, None, nonce); ber_ke_only = float(np.mean(em_e != m))
    # ---------------- receiver with K_d only: cannot decrypt, no valid header
    try:
        m_d, I_d, *_ = receiver(Is, Kd, Kd, nonce); kd_only = 'fails (wrong data)' if not np.array_equal(I_d, I) else 'UNEXPECTED'
    except Exception: kd_only = 'fails (no valid header)'
    if MESSAGE_SOURCE == 'text': open(f'{out}/recovered_message.txt', 'w').write(bits_to_bytes(m_rec).decode('utf-8', 'replace'))
    if MESSAGE_SOURCE == 'file': open(f'{out}/recovered_{os.path.basename(MESSAGE_FILE)}', 'wb').write(bits_to_bytes(m_rec))
    if MESSAGE_SOURCE == 'binary_image': Image.fromarray((m_rec.reshape(SECRET_IMAGE_SIZE[::-1]) * 255).astype(np.uint8)).save(f'{out}/recovered_secret.png')
    row = dict(image=name, size=f'{M}x{N}', message=mdesc, L=L, D=info['D'], rho=rho, excluded_blocks=info['n_excl_blocks'],
               header_bits=HDR_BITS, B_E_bits=info['len_E'], B_F_bits=info['len_F'], net_bits=len(m), net_bpp=len(m) / MN,
               gross_bpp=(HDR_BITS + info['len_E'] + info['len_F'] + len(m)) / MN,
               PSNR_marked=psnr(I, Im), SSIM_marked=ssim(I, Im), MSE_marked=mse(I, Im), max_pixel_change=int(np.abs(I.astype(int) - Im).max()),
               message_exact=exact_m, NC=nc(m, m_rec) if exact_m or len(m_rec) == len(m) else float('nan'),
               BER=float(np.mean(m_rec != m)) if len(m_rec) == len(m) else 1.0, cover_exact=exact_I, MSE_recovered=mse(I, I_rec),
               Ke_only_cover_exact=bool(np.array_equal(I_e, I)), Ke_only_message_BER=ber_ke_only, Kd_only=kd_only,
               t_sender_s=t_s, t_receiver_s=t_r)
    if SECURITY_METRICS:
        cp, ce = corr_all(I), corr_all(Is)
        row.update(H_plain=entropy(I), H_enc=entropy(Is), H_local_enc=local_entropy(Is), chi2_enc=chi2(Is),
                   **{f'corr_{k}_plain': cp[k] for k in 'HVD'}, **{f'corr_{k}_enc': ce[k] for k in 'HVD'})
        rng = np.random.default_rng(1); nps, uas = [], []
        for _ in range(NPCR_TRIALS):                                 # one-pixel change of the cover, same nonce
            I1 = I.copy(); i, j = rng.integers(0, M), rng.integers(0, N); I1[i, j] = (int(I1[i, j]) + 1) % 256
            try: Is1 = encrypt(embed(I1, m ^ payload_mask(Kd, len(m), nonce), L, rho=rho)[0], Ke, nonce)
            except (ValueError, RuntimeError): continue
            a, b = npcr_uaci(Is, Is1); nps.append(a); uas.append(b)
        a, b = npcr_uaci(Is, encrypt(Im, Ke, secrets.token_bytes(16)))  # fresh nonce
        row.update(NPCR=float(np.mean(nps)), UACI=float(np.mean(uas)), NPCR_fresh_nonce=a, UACI_fresh_nonce=b)
    json.dump({k: (v if not isinstance(v, (np.floating, np.integer, np.bool_)) else v.item()) for k, v in row.items()} | {'nonce_hex': nonce.hex()},
              open(f'{out}/result.json', 'w'), indent=1)
    if SAVE_FIGURES or SHOW_FIGURES: figure(name, I, Im, Is, I_rec, out)
    return row

def figure(name, I, Im, Is, Irec, out):
    fig, ax = plt.subplots(2, 4, figsize=(13, 5.6))
    for k, (A, t) in enumerate(((I, 'Cover'), (Im, f'Marked ({psnr(I, Im):.2f} dB)'), (Is, 'Encrypted'), (Irec, 'Recovered (exact)' if np.array_equal(Irec, I) else 'Recovered (ERROR)'))):
        ax[0, k].imshow(A, cmap='gray', vmin=0, vmax=255); ax[0, k].set_title(t); ax[0, k].axis('off')
        ax[1, k].bar(np.arange(256), np.bincount(A.ravel(), minlength=256), width=1, color='#2a78d6'); ax[1, k].set_xlim(0, 255); ax[1, k].set_yticks([])
    fig.suptitle(name); fig.tight_layout()
    if SAVE_FIGURES: fig.savefig(f'{out}/overview.png', dpi=150)
    if SHOW_FIGURES: plt.show()
    plt.close(fig)
    rng = np.random.default_rng(0); fig, ax = plt.subplots(2, 3, figsize=(10, 6.6))
    for r, (A, lab, col) in enumerate(((I.astype(float), 'Cover', '#2a78d6'), (Is.astype(float), 'Encrypted', '#eb6834'))):
        cc = corr_all(A)
        for k, (dy, dx, d) in enumerate(((0, 1, 'H'), (1, 0, 'V'), (1, 1, 'D'))):
            i = rng.integers(0, A.shape[0] - dy, 5000); j = rng.integers(0, A.shape[1] - dx, 5000)
            ax[r, k].scatter(A[i, j], A[i + dy, j + dx], s=1, color=col, alpha=0.5); ax[r, k].set_xlim(0, 255); ax[r, k].set_ylim(0, 255)
            ax[r, k].set_aspect('equal'); ax[r, k].set_title(f'{lab} {d}: r = {cc[d]:.4f}', fontsize=9)
    fig.tight_layout()
    if SAVE_FIGURES: fig.savefig(f'{out}/correlation.png', dpi=150)
    if SHOW_FIGURES: plt.show()
    plt.close(fig)

rows = []
for p in IMAGES:
    print(f'--- {os.path.basename(p)}'); r = process(p); rows.append(r)
    print(f"L = {r['L']}, D = {r['D']}, net = {r['net_bits']} bits ({r['net_bpp']:.4f} bpp), PSNR = {r['PSNR_marked']:.2f} dB, "
          f"message exact = {r['message_exact']}, cover exact = {r['cover_exact']}")
RESULTS = pd.DataFrame(rows); RESULTS.to_csv(os.path.join(OUTPUT_DIR, 'results.csv'), index=False)
print('\nresults written to', os.path.join(OUTPUT_DIR, 'results.csv'))

## 11. Summary

In [ ]:
cols = ['image', 'size', 'L', 'D', 'net_bits', 'net_bpp', 'PSNR_marked', 'SSIM_marked', 'message_exact', 'cover_exact', 'Ke_only_message_BER', 'Kd_only']
if SECURITY_METRICS: cols += ['H_enc', 'H_local_enc', 'chi2_enc', 'corr_H_enc', 'corr_V_enc', 'corr_D_enc', 'NPCR', 'UACI']
try: display(RESULTS[cols].round(4))
except NameError: print(RESULTS[cols].round(4).to_string())
print(f"\nall messages exact: {RESULTS.message_exact.all()} | all covers exact: {RESULTS.cover_exact.all()} | "
      f"K_e only never reveals the message: {(RESULTS.Ke_only_message_BER > 0.4).all() if len(RESULTS) and RESULTS.net_bits.min() > 1000 else 'n/a (short message)'}")
if SECURITY_METRICS and len(RESULTS):
    MN = int(np.median([int(s.split('x')[0]) * int(s.split('x')[1]) for s in RESULTS['size']])); n_c, u_c = npcr_critical(MN)
    print(f'NPCR/UACI acceptance at alpha = 0.05 for {MN} pixels: NPCR > {n_c:.4f} %, UACI in [{u_c[0]:.4f}, {u_c[1]:.4f}] %')
    print(RESULTS[['NPCR', 'UACI', 'NPCR_fresh_nonce', 'UACI_fresh_nonce']].describe().loc[['mean', 'min', 'max']].round(4).to_string())

## 12. Receiver only (independent check)

The receiver needs only the encrypted image, the public nonce and the keys. This cell reads the files written above from `OUTPUT_DIR` (or any folder with the same layout) and recovers the cover and the message without using anything else from the sender.

In [ ]:
RECEIVER_FOLDER = OUTPUT_DIR            # folder with one sub-folder per image containing encrypted.png and nonce.hex
for d in sorted(glob.glob(os.path.join(RECEIVER_FOLDER, '*', 'encrypted.png'))):
    sub = os.path.dirname(d); Is = np.asarray(Image.open(d)); nonce = bytes.fromhex(open(os.path.join(sub, 'nonce.hex')).read().strip())
    m_rec, I_rec, Im, em, hdr = receiver(Is, Ke, Kd, nonce)
    ok = np.array_equal(I_rec, np.asarray(Image.open(os.path.join(sub, 'cover_gray.png')))) if os.path.exists(os.path.join(sub, 'cover_gray.png')) else None
    msg = ''
    if MESSAGE_SOURCE == 'text': msg = ' | message: ' + bits_to_bytes(m_rec).decode('utf-8', 'replace')
    print(f"{os.path.basename(sub)}: header L = {hdr['L']}, D = {hdr['D']}, {hdr['len_M']} message bits; cover recovered exactly: {ok}{msg}")